# DARTH: Distance-Aware Real-Time Halting on HNSW

This notebook walks through the full DARTH pipeline:
1. Load siftsmall dataset
2. Build an `HNSW_DARTH` index
3. Collect training data (per-query features + recall label)
4. Train the LightGBM predictor
5. Compare Baseline HNSW vs DARTH across efSearch values
6. Recall–QPS trade-off plots
7. Per-query feature analysis (what does the predictor see?)

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from utils.read_files import read_fvecs, read_ivecs
from hsnw_constructionDARTH import HNSW_DARTH
from darth.collect_training_data import collect_training_data
from darth.train_predictor import train as train_predictor
from darth.predictor import LGBMPredictor

print("Imports OK")

## 1. Load siftsmall

In [ ]:
BASE_DIR = os.path.join(ROOT, "Datasets", "siftsmall")

xb = read_fvecs(os.path.join(BASE_DIR, "siftsmall_base.fvecs"))
xq = read_fvecs(os.path.join(BASE_DIR, "siftsmall_query.fvecs"))
gt = read_ivecs(os.path.join(BASE_DIR, "siftsmall_groundtruth.ivecs"))

# Use part of the base set as learn set for training data collection
xlearn = xb[:5000]

print(f"base  : {xb.shape}")
print(f"query : {xq.shape}")
print(f"gt    : {gt.shape}")
print(f"learn : {xlearn.shape}")

## 2. Compute exact ground truth for the learn set

In [ ]:
def exact_gt(xb, xq, k=10):
    """Brute-force L2 ground truth (batched to avoid large allocs)."""
    gt = np.empty((len(xq), k), dtype=np.int32)
    batch = 500
    for start in range(0, len(xq), batch):
        end = min(start + batch, len(xq))
        q = xq[start:end].astype(np.float32)
        diffs = (q**2).sum(1, keepdims=True) + (xb**2).sum(1) - 2.0*(q @ xb.T)
        gt[start:end] = np.argsort(diffs, axis=1)[:, :k]
    return gt

k = 10
print(f"Computing exact GT for {len(xlearn)} learn queries …")
gt_learn = exact_gt(xb, xlearn, k=k)
print(f"gt_learn: {gt_learn.shape}")

## 3. Build HNSW_DARTH index

In [ ]:
M   = 16
efC = 200

print(f"Building index (M={M}, efC={efC}, n={len(xb)}) …")
t0 = time.time()
hnsw = HNSW_DARTH(dim=xb.shape[1], M=M, efConstruction=efC, metric="l2")
for i, v in enumerate(xb):
    hnsw._insert_(v, i)
    if (i+1) % 2000 == 0:
        print(f"  {i+1}/{len(xb)}", flush=True)
print(f"Built in {time.time()-t0:.1f}s")

## 4. Collect training data

For each learn query, we run HNSW with full `efSearch`, and at each prediction
checkpoint record: `(features, recall_achieved)`.  This is the supervised signal
the LightGBM model learns from.

In [ ]:
os.makedirs(os.path.join(ROOT, "predictor_models"), exist_ok=True)
TRAIN_CSV = os.path.join(ROOT, "predictor_models", "train_data_siftsmall_M16_efC200_k10.csv")

if os.path.exists(TRAIN_CSV):
    print(f"CSV already exists: {TRAIN_CSV}")
else:
    collect_training_data(
        hnsw, xlearn, gt_learn,
        k=k,
        efSearch=efC,
        output_path=TRAIN_CSV,
        logging_interval=2,
        verbose=True,
    )
    print("Done.")

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print(f"Training rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

### Feature distributions

In [ ]:
feature_cols = [c for c in df.columns if c != "label"]
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=40, color="steelblue", edgecolor="white", linewidth=0.3)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("value")
    axes[i].set_ylabel("count")
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle("DARTH training feature distributions (siftsmall)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Label distribution: fraction of checkpoints where recall >= Rt
Rt = 0.90
if "label" in df.columns:
    pos = (df["label"] >= Rt).mean()
    print(f"Fraction of checkpoints with recall >= {Rt}: {pos:.3f}")
    df["label"].hist(bins=30, figsize=(7,3), color="coral", edgecolor="white")
    plt.axvline(Rt, color="red", linestyle="--", label=f"Rt={Rt}")
    plt.xlabel("recall at checkpoint")
    plt.ylabel("count")
    plt.title("Label distribution (recall achieved at each DARTH checkpoint)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 5. Train LightGBM predictor

In [ ]:
MODEL_TXT = os.path.join(ROOT, "predictor_models", "siftsmall_M16_efC200_k10.txt")

if os.path.exists(MODEL_TXT):
    print(f"Model already exists: {MODEL_TXT}")
else:
    train_predictor(
        input_csv=TRAIN_CSV,
        output_model=MODEL_TXT,
        n_estimators=100,
        val_fraction=0.1,
        verbose=True,
    )

predictor = LGBMPredictor(MODEL_TXT)
print("Predictor loaded.")

## 6. Baseline HNSW vs DARTH — sweep over efSearch

For each `efSearch` value we measure:
- **Recall@10** — fraction of true neighbours found
- **QPS** — queries per second

This traces a recall–QPS curve for each method.

In [ ]:
def recall_at_k(I, gt, k):
    hits = sum(len(set(I[i,:k].tolist()) & set(gt[i,:k].tolist())) for i in range(len(I)))
    return hits / (len(I) * k)

efSearch_values = [10, 20, 40, 80, 100, 150, 200]
Rt    = 0.90
ipi   = 200
mpi   = 20

results = []
for ef in efSearch_values:
    # ── Baseline ──────────────────────────────────────────────────────────────
    t0 = time.time()
    D_b, I_b = hnsw.search_darth(xq, k=k, efSearch=ef, Rt=0.0,
                                  predictor=None, ipi=ef+1, mpi=ef+1)
    t_base = time.time() - t0
    r_base = recall_at_k(I_b, gt, k)

    # ── DARTH ────────────────────────────────────────────────────────────────
    t0 = time.time()
    D_d, I_d = hnsw.search_darth(xq, k=k, efSearch=ef, Rt=Rt,
                                  predictor=predictor, ipi=ipi, mpi=mpi)
    t_darth = time.time() - t0
    r_darth = recall_at_k(I_d, gt, k)

    nq = len(xq)
    results.append(dict(
        ef=ef,
        r_base=r_base,  qps_base=nq/t_base,
        r_darth=r_darth, qps_darth=nq/t_darth,
        speedup=t_base/t_darth if t_darth > 0 else 0,
    ))
    print(f"ef={ef:>4}  Base R@{k}={r_base:.4f}  DARTH R@{k}={r_darth:.4f}  "
          f"Speedup={t_base/t_darth:.2f}x")

res = pd.DataFrame(results)
res

### Recall–QPS trade-off curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── left: Recall–QPS ─────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(res["qps_base"],  res["r_base"],  "o-",  label="Baseline HNSW",  color="steelblue")
ax.plot(res["qps_darth"], res["r_darth"], "s--", label=f"DARTH (Rt={Rt})", color="tomato")
for _, row in res.iterrows():
    ax.annotate(f"ef={int(row['ef'])}", (row["qps_base"],  row["r_base"]),
                textcoords="offset points", xytext=(4, 4), fontsize=7, color="steelblue")
    ax.annotate(f"ef={int(row['ef'])}", (row["qps_darth"], row["r_darth"]),
                textcoords="offset points", xytext=(4, -10), fontsize=7, color="tomato")
ax.axhline(Rt, color="grey", linestyle=":", linewidth=0.8, label=f"Target Rt={Rt}")
ax.set_xlabel("QPS (queries / second)", fontsize=11)
ax.set_ylabel(f"Recall@{k}", fontsize=11)
ax.set_title("Recall–QPS trade-off (siftsmall)", fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# ── right: Speedup vs efSearch ────────────────────────────────────────────────
ax2 = axes[1]
bars = ax2.bar(res["ef"].astype(str), res["speedup"], color="mediumseagreen", edgecolor="white")
ax2.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
for bar, val in zip(bars, res["speedup"]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.2f}x", ha="center", va="bottom", fontsize=9)
ax2.set_xlabel("efSearch", fontsize=11)
ax2.set_ylabel("Speedup (baseline / DARTH)", fontsize=11)
ax2.set_title("DARTH Speedup per efSearch", fontsize=12)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(ROOT, "figures", "darth_recall_qps.pdf"), bbox_inches="tight")
plt.show()

## 7. Failure case analysis

We identify queries where DARTH terminates early but **fails to meet Rt**.
These are the *false positives* of the predictor — it predicted "good enough"
but recall was actually below Rt.

In [ ]:
# Run full efSearch with both methods, collect per-query recall
efSearch = 200

_, I_base_full = hnsw.search_darth(xq, k=k, efSearch=efSearch, Rt=0.0,
                                    predictor=None, ipi=efSearch+1, mpi=efSearch+1)
_, I_darth_full = hnsw.search_darth(xq, k=k, efSearch=efSearch, Rt=Rt,
                                     predictor=predictor, ipi=ipi, mpi=mpi)

# Per-query recall
r_base_q  = np.array([len(set(I_base_full[i,:k])  & set(gt[i,:k])) / k for i in range(len(xq))])
r_darth_q = np.array([len(set(I_darth_full[i,:k]) & set(gt[i,:k])) / k for i in range(len(xq))])

# Failure: DARTH recall < Rt AND baseline recall >= Rt  (early stop hurt this query)
failure_mask = (r_darth_q < Rt) & (r_base_q >= Rt)
n_fail = failure_mask.sum()
print(f"Queries where DARTH fails (recall < {Rt}) but baseline succeeds: {n_fail}/{len(xq)} ({100*n_fail/len(xq):.1f}%)")

In [ ]:
# Scatter: baseline recall vs DARTH recall per query
fig, ax = plt.subplots(figsize=(7, 6))

ok_mask = ~failure_mask
ax.scatter(r_base_q[ok_mask],     r_darth_q[ok_mask],     alpha=0.3, s=10,
           color="steelblue",  label="Normal queries")
ax.scatter(r_base_q[failure_mask], r_darth_q[failure_mask], alpha=0.8, s=25,
           color="red",        label=f"DARTH failures ({n_fail})")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="y=x")
ax.axhline(Rt, color="grey", linestyle=":", linewidth=0.8, label=f"Rt={Rt}")
ax.set_xlabel("Baseline Recall@10", fontsize=11)
ax.set_ylabel("DARTH Recall@10",    fontsize=11)
ax.set_title("Per-query recall: Baseline vs DARTH", fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "figures", "darth_failure_scatter.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Characterise failing queries — what makes them hard?
fail_idx = np.where(failure_mask)[0]

# Norm of failing queries vs all queries
q_norms_all  = np.linalg.norm(xq, axis=1)
q_norms_fail = q_norms_all[fail_idx]

# Distance to nearest neighbour in ground truth (proxy for query difficulty)
nn_dists_all  = np.array([
    np.linalg.norm(xq[i] - xb[gt[i, 0]]) for i in range(len(xq))
])
nn_dists_fail = nn_dists_all[fail_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(q_norms_all,  bins=30, alpha=0.5, label="All queries",     color="steelblue", density=True)
axes[0].hist(q_norms_fail, bins=30, alpha=0.7, label="Failing queries", color="red",       density=True)
axes[0].set_xlabel("Query L2 norm", fontsize=11)
axes[0].set_ylabel("Density")
axes[0].set_title("Query norm distribution")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(nn_dists_all,  bins=30, alpha=0.5, label="All queries",     color="steelblue", density=True)
axes[1].hist(nn_dists_fail, bins=30, alpha=0.7, label="Failing queries", color="red",       density=True)
axes[1].set_xlabel("Distance to true 1-NN", fontsize=11)
axes[1].set_ylabel("Density")
axes[1].set_title("Query difficulty (1-NN distance)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Characteristics of DARTH failure queries vs all queries", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "figures", "darth_failure_analysis.pdf"), bbox_inches="tight")
plt.show()

print(f"Mean 1-NN dist  — all: {nn_dists_all.mean():.2f}  failures: {nn_dists_fail.mean():.2f}")
print(f"Mean query norm — all: {q_norms_all.mean():.2f}   failures: {q_norms_fail.mean():.2f}")

## 8. Predictor feature importance

In [ ]:
try:
    import lightgbm as lgb
    booster = lgb.Booster(model_file=MODEL_TXT)
    importance = pd.Series(
        booster.feature_importance(importance_type="gain"),
        index=booster.feature_name()
    ).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    importance.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
    ax.set_xlabel("Feature importance (gain)", fontsize=11)
    ax.set_title("LightGBM predictor — feature importance", fontsize=12)
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, "figures", "darth_feature_importance.pdf"), bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Could not load booster for importance plot: {e}")

## 9. DARTH vs Baseline summary table

In [ ]:
print(f"{'efSearch':>8} | {'Base R@10':>9} {'Base QPS':>9} | {'DARTH R@10':>10} {'DARTH QPS':>10} | {'Speedup':>7}")
print("-" * 68)
for _, row in res.iterrows():
    print(f"{int(row['ef']):>8} | {row['r_base']:>9.4f} {row['qps_base']:>9.1f} | "
          f"{row['r_darth']:>10.4f} {row['qps_darth']:>10.1f} | {row['speedup']:>7.2f}x")